# Deep Hedging — Phase 1 vers phase 2, brique 4 : coûts et mesure de risque

On rallume les coûts de transaction, et tout bascule. Sans coûts, rééquilibrer plus souvent était toujours mieux (l'erreur en 1/sqrt(n)). Avec coûts, trader souvent coûte cher, donc il existe une **fréquence optimale**, et le delta-hedging cesse d'être la stratégie parfaite.

Pour parler d'« optimal », il faut un **score unique** d'une distribution de P&L. On introduit ici la mesure de risque **CVaR**, qui sera aussi la fonction objectif du réseau en phase 3.

In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
RNG = np.random.default_rng(0)

## Briques précédentes (notebook autonome), avec le coût déjà branché

In [ ]:
def simulate_gbm(S0, mu, sigma, T, n_steps, n_paths, rng=RNG):
    dt = T/n_steps
    Z = rng.standard_normal((n_paths, n_steps))
    inc = (mu - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z
    return S0*np.exp(np.concatenate([np.zeros((n_paths,1)), np.cumsum(inc,axis=1)], axis=1))

def bs_price(S, K, tau, r, sigma):
    S = np.asarray(S, float)
    d1 = (np.log(S/K)+(r+0.5*sigma**2)*tau)/(sigma*np.sqrt(tau)); d2 = d1 - sigma*np.sqrt(tau)
    return S*norm.cdf(d1) - K*np.exp(-r*tau)*norm.cdf(d2)

def bs_delta(S, K, tau, r, sigma):
    S = np.asarray(S, float)
    return norm.cdf((np.log(S/K)+(r+0.5*sigma**2)*tau)/(sigma*np.sqrt(tau)))

def delta_hedge_pnl(S, K, T, r, sigma, cost=0.0):
    m, n1 = S.shape; n = n1-1; dt = T/n; times = np.linspace(0,T,n1)
    cash = bs_price(S[:,0],K,T,r,sigma).copy(); sh = np.zeros(m)
    for k in range(n):
        tau = T-times[k]; dk = bs_delta(S[:,k],K,tau,r,sigma); tr = dk-sh
        cash -= tr*S[:,k]; cash -= cost*np.abs(tr)*S[:,k]; sh = dk; cash *= np.exp(r*dt)
    return cash + sh*S[:,-1] - np.maximum(S[:,-1]-K, 0.0)

## La CVaR : scorer toute une distribution de P&L par un seul nombre

La perte est `L = -P&L`. La **VaR** au niveau alpha est le quantile `alpha` de la perte (le seuil que la perte ne dépasse qu'avec probabilité `1-alpha`). La **CVaR** (expected shortfall) est la perte **moyenne au-delà** de ce seuil : la moyenne des pires `(1-alpha)` scénarios.

    CVaR_alpha(L) = E[ L | L >= VaR_alpha(L) ]

Plus la CVaR est basse, meilleure est la stratégie. C'est une mesure de risque *cohérente* (elle récompense la diversification), contrairement à la VaR, et elle admet une forme minimisable (Rockafellar-Uryasev) qu'on réutilisera pour entraîner le réseau :

    CVaR_alpha(L) = min_w { w + (1/(1-alpha)) E[(L - w)+] }

In [ ]:
def cvar(pnl, alpha=0.95):
    """CVaR de la perte L = -pnl : moyenne des pires (1-alpha) scénarios."""
    loss = -pnl
    var = np.quantile(loss, alpha)
    return loss[loss >= var].mean()

## Le compromis de fréquence

On mesure, pour plusieurs niveaux de coût, comment le P&L moyen, l'écart-type et la CVaR évoluent avec la fréquence de rééquilibrage. Le P&L moyen se dégrade avec le trading (coûts), l'écart-type baisse (réplication), et la CVaR combine les deux.

In [ ]:
S0, K, mu, r, sigma, T = 100., 100., 0.10, 0.02, 0.20, 1.0
m = 100_000
ns = [5, 10, 21, 42, 63, 126, 252]

for cost in [0.0, 0.005, 0.02]:
    print(f"=== coût = {cost:.1%} ===")
    print(f"{'n_steps':>8} | {'moy P&L':>9} | {'std':>7} | {'CVaR95':>8}")
    for n in ns:
        S = simulate_gbm(S0, mu, sigma, T, n, m)
        pnl = delta_hedge_pnl(S, K, T, r, sigma, cost)
        print(f"{n:>8} | {pnl.mean():>9.3f} | {pnl.std():>7.3f} | {cvar(pnl):>8.3f}")
    print()

## Visualisation : la forme en U

Sans coût, la CVaR baisse toujours. Avec coût, elle a un **minimum** (cercle) à une fréquence finie, et ce minimum se déplace vers la gauche quand le coût monte. Le message : avec des frictions, il n'y a plus de couverture parfaite, seulement un meilleur compromis, et le delta-hedging est bloqué sur la règle du delta.

In [ ]:
ns = [5, 10, 21, 42, 63, 126, 252]
fig, ax = plt.subplots(figsize=(7, 5))
for cost, c in [(0.0, "green"), (0.005, "orange"), (0.02, "crimson")]:
    cs = [cvar(delta_hedge_pnl(simulate_gbm(S0, mu, sigma, T, n, m), K, T, r, sigma, cost)) for n in ns]
    ax.plot(ns, cs, "o-", color=c, label=f"coût={cost:.1%}")
    if cost > 0:
        i = int(np.argmin(cs))
        ax.scatter([ns[i]], [cs[i]], s=140, facecolors="none", edgecolors=c, linewidths=2)
ax.set_xscale("log"); ax.set_xlabel("n_steps"); ax.set_ylabel("CVaR 95% de la perte")
ax.set_title("Coûts : le compromis de fréquence (cercle = optimum)")
ax.legend(); ax.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()

## Ce que ça prépare

Le delta-hedging ne peut que choisir *quand* rééquilibrer vers le delta. Un réseau, lui, pourra choisir *combien* détenir à chaque instant, en fonction de l'état complet (prix, temps restant, position courante), pour minimiser directement la CVaR sous coûts. Il pourra apprendre, par exemple, à ne pas trader tant que le delta n'a pas assez bougé (une bande de non-transaction), ce que la règle du delta ne sait pas faire. C'est l'objet de la phase 3.